<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T01-L01-statistics-by-simulation/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T01-L01-statistics-by-simulation/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T01-L01-statistics-by-simulation/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy, which Colab, Kaggle, Binder and
Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T01-L01-statistics-by-simulation/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T01-L01 · Statistics by simulation: sampling distributions, bootstrap intervals, permutation tests, power, peeking

**You will build:** the working kit of an experiment analyst, from numpy alone — a simulator of
sampling distributions, a bootstrap interval and a check of how often it is right, a
permutation test, a simulation-based power and sample-size calculation, and two audits that
measure how often peeking at a running test, and testing many metrics at once, hand you a
false positive.

**Time:** ~90 minutes · **Runs on:** a laptop CPU, 8 GiB RAM, no GPU, no download ·
**Prerequisites:** T00-L01-the-8gb-track.

Every "5%" in an A/B-test readout is a promise about what would happen if you repeated the
experiment many times. A computer can repeat an experiment many times, so instead of trusting a
formula you can **watch** the promise being kept or broken. This lesson does that for the
claims that decide launches: that a 95% interval covers the truth 95% of the time, that a
p-value of 0.05 is a 5% false-alarm rate, that a test has the power you planned, and that
looking at the data early or testing twenty things costs you nothing. Every number the lesson
prints is computed by the code you run.

By the end you will be able to:

1. Implement a simulator of sampling distributions and measure how the standard error of a mean
   and of a median shrink with sample size for a heavy-tailed population.
2. Implement a percentile bootstrap interval and measure how often it covers the truth,
   against its nominal level.
3. Implement a permutation test for a difference in means and an estimator of statistical power
   by simulation, and use them to choose a sample size.
4. Measure the false-positive rate of a test that is peeked at repeatedly, and of twenty
   metrics tested at once, and implement the Bonferroni and Holm corrections that repair it.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T00-L01`

**This lesson assumes you can already:**

- write Python functions, including one that takes another function as an argument and calls it
- use numpy arrays: read a shape, take a mean or a sum along an axis, and compare an array with a number to get a boolean array
- say in words what a mean, a median and a standard deviation measure, and what a histogram shows
- read "p < 0.05" or "95% confidence interval" in a report, even if you could not yet say exactly what it promises

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Think Stats, 3rd edition (chapters on Estimation and Hypothesis Testing)](https://allendowney.github.io/ThinkStats/) · Allen B. Downey · *free* — a Python book that teaches sampling distributions, confidence intervals and tests by simulation; the closest match to this lesson's method
- [Statistical Inference via Data Science: Bootstrapping and Confidence Intervals, Hypothesis Testing](https://moderndive.com/) · Chester Ismay and Albert Y. Kim (ModernDive) · *free* — chapters 8 and 9 build the bootstrap and permutation-style tests from shuffling and resampling; the code is in R but the pictures carry over to exercises 3 and 5
- [NumPy: the absolute basics for beginners](https://numpy.org/doc/stable/user/absolute_beginners.html) · NumPy documentation · *free* — shapes, axes and boolean comparisons, which every exercise here uses
- [Computer Age Statistical Inference: Algorithms, Evidence and Data Science](https://hastie.su.domains/CASI/) · Bradley Efron and Trevor Hastie · *free* — a PDF the authors offer free for personal use, with the publisher's permission; its chapters on the jackknife, the bootstrap and bootstrap confidence intervals go further than this lesson does on exercises 3 and 4

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import itertools
import math
import os
import signal
import sys
import threading
import time
import traceback
from typing import Callable, Sequence

import numpy as np

print("python", sys.version.split()[0], "· numpy", np.__version__, "· platform", sys.platform)


# A time limit for every check that calls the student's code: a loop that never ends must fail
# its check, not freeze the notebook or the grader (which imports the notebook). Copy this block
# unchanged; change only the message. Why it is this careful: whoever runs the notebook may
# already have an alarm running (tools/grade.py stops every rubric row after 45 s and repeats
# its alarm every 0.25 s; a harness may have an alarm that ends the process). Earlier guards
# switched that alarm off on the way out (`setitimer(ITIMER_REAL, 0)`), so the NEXT hang ran for
# ever. This one saves the running alarm, lets whichever deadline comes first win (handing the
# signal to the outer handler when it is the outer's turn), and on the way out re-arms the outer
# alarm with the time it had left and its interval. tools/verify_all.py fails a lesson whose
# guard does less.
# How long an exercise's check may run before it counts as stuck, and how long any other cell that runs your
# code (a demonstration, a table) may: a right answer needs a small fraction of either, measured on the worked
# solution (internal/retrofit_guard.py, 10x or more). tools/verify_all.py reads both numbers.
_CHECK_SECONDS = 2
_DEMO_SECONDS = 60


class _OutOfTime(BaseException):
    """Raised inside the guarded block when its time is up. A BaseException, so `except
    Exception:` in the code under test cannot swallow it; it repeats every 0.25 s in case a bare
    `except:` does. `_time_limit` turns it into an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str, memory_mib: float = 2048):
    """Fail the block with an AssertionError naming `what` if it runs longer than `seconds`.
    Uses the operating system's alarm (Linux and macOS, main thread); a no-op elsewhere."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped block's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        import gc
        gc.collect()                                     # as the growth of this block
    message = (f"{what} was still running after {seconds:g} s: a loop whose stopping condition "
               "is never met runs for ever; give every loop a cap")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and not state["fired"]:
            due.append(now + 0.1)                        # and look at the memory every 0.1 s
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and not state["fired"] else 0
        if grown > memory_mib and state["next"] is not None:       # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_time_limit", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)

    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


SEED = 20261002           # this lesson's one fixed seed. Every random stream below is derived
                           # from it and a short label (make_rng), so re-running prints the
                           # same results on any machine, and no experiment shares a stream
                           # with another.
ALPHA = 0.05              # the nominal level used throughout: "5%".
LEVEL = 0.95              # the nominal coverage of every confidence interval: "95%".

_FAILED_CHECKS: list[str] = []
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run
_RESULTS: dict[str, float] = {}  # headline numbers the demos measured, for the closing scorecard

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("simulate_sampling_distribution",),
    "exercise 2": ("empirical_se", "loglog_slope"),
    "exercise 3": ("bootstrap_distribution", "percentile_interval"),
    "exercise 4": ("coverage_rate",),
    "exercise 5": ("permutation_test",),
    "exercise 6": ("estimate_power",),
    "exercise 7": ("smallest_n_for_power",),
    "exercise 8": ("z_statistics_at_looks",),
    "exercise 9": ("rejection_rates",),
    "exercise 10": ("bonferroni_reject", "holm_reject"),
}


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (bm25_rank)"; several -> "exercises 3, 6 and 7"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = (),
         limit: float = 0) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero. Every check also has a time limit (`limit` seconds), so a loop that never
    ends in your code fails its check instead of freezing the notebook.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        with _time_limit(limit or (_CHECK_SECONDS if label in _EXERCISES else _DEMO_SECONDS), f"{label}'s check"):
            check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"


def _demo(label: str, fn: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """A demonstration cell: `_try`, except that it only runs when this file is the program
    being run (a notebook, or `python lesson.py`), never when the autograder imports it just to
    call your functions. A demo gets a longer time limit than a check."""
    if __name__ == "__main__":
        _try(label, fn, needs)


_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")

## 1. The population, and a way to draw from it (given)

Product metrics such as revenue per user, session length and latency are often far from
bell-shaped: most users contribute a little and a few contribute a great deal. This lesson's
population is a **Pareto** distribution, a standard model of that shape. Its smallest possible value is
`PARETO_XM`; its tail index `PARETO_ALPHA` sets how heavy the tail is (smaller is heavier).
We chose a tail index above 2, so the population has a finite mean and a finite variance —
every textbook formula applies — but the tail is still heavy enough that the textbook
approximations arrive late. That gap is what this lesson measures.

Because we built the population, we know the truth: its mean, median and standard deviation
have closed forms (`TRUE_MEAN`, `TRUE_MEDIAN`, `TRUE_SD`). A simulation can therefore be graded
against the thing it estimates, which you can almost never do with real data.

**Randomness in this lesson.** Every function you write takes an `rng`, a numpy `Generator`,
and draws all its randomness from it. `make_rng("label", n)` builds an independent, reproducible
generator from the lesson seed, so two experiments never share a stream and re-running gives
identical output. Draws come from `rng.random` and `rng.integers` only, the simplest numpy
draws, because numpy reserves the right to change a Generator's stream between feature
releases (see `claims.yaml`). This lesson is checked to print the same numbers on two numpy
versions; a future numpy may not. Read this section; nothing in it is graded.

In [ ]:
PARETO_ALPHA = 2.5        # tail index: smaller means a heavier tail (mean needs > 1, variance > 2).
PARETO_XM = 1.0           # the smallest value the population can produce.

TRUE_MEAN = PARETO_ALPHA * PARETO_XM / (PARETO_ALPHA - 1.0)
TRUE_MEDIAN = PARETO_XM * 2.0 ** (1.0 / PARETO_ALPHA)
TRUE_SD = PARETO_XM * math.sqrt(PARETO_ALPHA / ((PARETO_ALPHA - 1.0) ** 2 * (PARETO_ALPHA - 2.0)))


def make_rng(*key: object) -> np.random.Generator:
    """An independent, reproducible generator for the experiment named by `key` (given)."""
    ints = [SEED] + [int(k) if isinstance(k, (int, np.integer)) else
                     sum(ord(c) * 131 ** i for i, c in enumerate(str(k))) % (2 ** 32)
                     for k in key]
    return np.random.default_rng(ints)


def draw_population(rng: np.random.Generator, size) -> np.ndarray:
    """`size` draws from the heavy-tailed population (given): inverse-CDF sampling of a Pareto,
    from `rng.random` alone."""
    u = rng.random(size)
    return PARETO_XM * (1.0 - u) ** (-1.0 / PARETO_ALPHA)


def welch_z(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """The Welch two-sample z statistic, (mean x - mean y) / sqrt(var x / n_x + var y / n_y),
    computed along the LAST axis, so a matrix of experiments gives one z per row (given).
    Variances use the n-1 divisor."""
    nx, ny = x.shape[-1], y.shape[-1]
    return (x.mean(axis=-1) - y.mean(axis=-1)) / np.sqrt(
        x.var(axis=-1, ddof=1) / nx + y.var(axis=-1, ddof=1) / ny)


def two_sided_p(z: np.ndarray) -> np.ndarray:
    """The two-sided p-value of a z statistic under a standard normal null (given), from the
    complementary error function: p = erfc(|z| / sqrt 2)."""
    flat = np.abs(np.asarray(z, dtype=float)).ravel()
    out = np.array([math.erfc(v / math.sqrt(2.0)) for v in flat])
    return out.reshape(np.shape(z))


def normal_critical(alpha: float) -> float:
    """The z value whose two-sided normal tail probability is `alpha` (given), found by
    bisection on `two_sided_p`'s own formula, so no statistics library is needed."""
    lo, hi = 0.0, 12.0
    for _ in range(200):
        mid = (lo + hi) / 2.0
        if math.erfc(mid / math.sqrt(2.0)) > alpha:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2.0


def _show_population() -> None:
    rng = make_rng("population")
    big = draw_population(rng, 1_000_000)
    print(f"population: Pareto, smallest value {PARETO_XM:g}, tail index {PARETO_ALPHA:g}")
    print(f"closed forms:  mean {TRUE_MEAN:.4f}   median {TRUE_MEDIAN:.4f}   sd {TRUE_SD:.4f}")
    print(f"{big.size:,} simulated draws: mean {big.mean():.4f}   median {np.median(big):.4f}"
          f"   sd {big.std(ddof=1):.4f}")
    order = np.sort(big)
    top = order[-big.size // 100:].sum() / order.sum()
    print(f"the top 1% of draws carry {100 * top:.1f}% of the total; the largest single draw is "
          f"{order[-1]:.1f}, {order[-1] / TRUE_MEAN:.0f} times the mean.")
    small = draw_population(make_rng("population", "ten"), 10)
    print("ten draws:", np.round(small, 2).tolist())
    print(f"their mean is {small.mean():.3f} and their median {np.median(small):.3f}; the truth "
          f"is {TRUE_MEAN:.3f} and {TRUE_MEDIAN:.3f}. One sample is one draw from a sampling "
          "distribution, and the next section looks at that distribution.")


_show_population()

## 2. Exercise 1 — a sampling distribution, by brute force

A **sampling distribution** is the answer to "if I ran this study again and again, what would
my statistic be each time?". You never get to see it for real, because you only run the study
once. Here you can: draw `reps` independent samples of size `n`, compute the statistic on each,
and the `reps` numbers you get *are* the sampling distribution, as closely as `reps` allows.

Your function receives the statistic as a function. To keep it fast it is called ONCE, on a
whole matrix with one sample per row, as `statistic(samples, axis=1)`, and returns one number
per row — which is exactly how `np.mean` and `np.median` already behave.

<details><summary>💡 Hint 1 — what to think about</summary>

How many numbers do you need to draw in total? One independent sample per replicate means
`reps` rows and `n` columns, all fresh. Which given function draws from the population, and
which argument of the statistic says "one answer per row"?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Draw all the observations in a single call, shaped as a matrix of `reps` rows by `n` columns,
from the `rng` you were handed (never from a new generator: the caller controls the seed).
Then hand that matrix to the statistic with the row-wise axis and return what it gives back.
</details>

In [ ]:
def simulate_sampling_distribution(statistic: Callable[..., np.ndarray], n: int, reps: int,
                                    rng: np.random.Generator) -> np.ndarray:
    """The sampling distribution of `statistic` at sample size `n`, from `reps` fresh samples.

    Requirements, each of which is graded:
      * `reps` INDEPENDENT samples, each of exactly `n` draws from `draw_population`, all taken
        from the `rng` you were given (it is the caller's, so it must be advanced by your draws).
      * `statistic` is called once, as `statistic(samples, axis=1)`, on the `(reps, n)` matrix;
        return what it computes as a float array of shape `(reps,)`.

    Example:
        >>> out = simulate_sampling_distribution(np.mean, 5, 3, make_rng("doc"))
        >>> out.shape
        (3,)
        >>> bool(out.min() >= PARETO_XM)   # a mean of draws cannot go below the smallest draw
        True
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_simulate_sampling_distribution() -> None:
    out = simulate_sampling_distribution(np.mean, 8, 12, make_rng("check", 1))
    assert out.shape == (12,), f"expected one value per replicate, shape (12,), got {out.shape}"
    assert out.min() >= PARETO_XM, (
        "a mean of draws from the population cannot be below its smallest value: draw with "
        "draw_population"
    )

    def ones_per_row(a, axis):
        return np.sum(np.ones_like(a), axis=axis)       # counts the draws in each sample

    lengths = simulate_sampling_distribution(ones_per_row, 8, 12, make_rng("check", 2))
    assert np.all(lengths == 8), (
        f"every sample must have exactly n = 8 draws, but the statistic saw {np.unique(lengths)}"
    )

    def first_draw(a, axis):
        return np.take(a, 0, axis=axis)

    firsts = simulate_sampling_distribution(first_draw, 8, 12, make_rng("check", 3))
    assert len(np.unique(firsts)) == 12, (
        "the 12 replicates must be 12 INDEPENDENT samples: draw a fresh (reps, n) matrix, "
        "do not reuse one sample"
    )
    a = simulate_sampling_distribution(np.mean, 8, 12, make_rng("check", 4))
    b = simulate_sampling_distribution(np.mean, 8, 12, make_rng("check", 4))
    c = simulate_sampling_distribution(np.mean, 8, 12, make_rng("check", 5))
    assert np.array_equal(a, b) and not np.array_equal(a, c), (
        "the result must depend on the rng you are given (same rng state, same answer; "
        "another state, another answer) — draw from `rng`, never from a new generator"
    )
    print("exercise 1 looks right — one independent sample of n draws per replicate, "
          "all drawn from the rng you were given")

In [ ]:
_try("exercise 1", _check_simulate_sampling_distribution)

## 3. Exercise 2 — how fast does the standard error shrink?

The **standard error** of a statistic is the standard deviation of its sampling distribution:
how far the statistic typically lands from where it is centred, from one repeat of the study to
the next. Textbooks say the standard error of a mean falls like `1 / sqrt(n)`, so quadrupling
the sample size halves it. On a log-log plot a power law `SE = c * n^k` is a straight line with
slope `k`, so the fastest honest way to check the textbook is to fit that slope.

Two small functions. `empirical_se` is the standard deviation of a simulated sampling
distribution — and because it is a sample, it uses the unbiased `n - 1` divisor.
`loglog_slope` fits the straight line, by least squares and WITH an intercept, because `c` is
not 1.

<details><summary>💡 Hint 1 — what to think about</summary>

A sample standard deviation divides by `n - 1`, not `n`. Numpy's `std` defaults to the other
one: which argument changes that? For the slope: take logs of BOTH the sizes and the values
(same base), then ask what a fitted line `log y = k * log x + c` needs besides `k`.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Least squares for a line with an intercept: centre both logged arrays on their own means, and
the slope is the sum of the products of the centred values divided by the sum of the squared
centred sizes. A library routine that fits a degree-one polynomial does the same in one call.
</details>

In [ ]:
def empirical_se(values: Sequence[float]) -> float:
    """The standard error estimated from a simulated sampling distribution: the sample standard
    deviation of `values`, with the unbiased `n - 1` divisor.

    Example:
        >>> round(empirical_se([1.0, 2.0, 3.0, 4.0]), 4)
        1.291
    """
    # YOUR CODE HERE
    raise NotImplementedError


def loglog_slope(ns: Sequence[float], values: Sequence[float]) -> float:
    """The slope `k` of the least-squares line through `(log n, log value)`, with an intercept.

    A quantity that follows `value = c * n**k` has slope `k` whatever `c` is.

    Example:
        >>> round(loglog_slope([10, 100, 1000], [3 * 10 ** -0.5, 3 * 100 ** -0.5, 3 * 1000 ** -0.5]), 6)
        -0.5
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_se_and_slope() -> None:
    se = empirical_se([1.0, 2.0, 3.0, 4.0])
    assert abs(se - 1.2909944487358056) < 1e-9, (
        f"the standard deviation of 1, 2, 3, 4 with the n-1 divisor is 1.2910, got {se:.4f} — "
        "numpy's std divides by n unless you pass ddof=1"
    )
    ns = [10, 40, 160, 640]
    k = loglog_slope(ns, [3.0 * n ** -0.5 for n in ns])
    assert abs(k - (-0.5)) < 1e-9, (
        f"values that are exactly 3 * n**-0.5 have log-log slope -0.5, got {k:.4f} — fit a line "
        "WITH an intercept (the constant 3 is not 1), and take the log of both columns"
    )
    k2 = loglog_slope([1, 10, 100], [5.0, 50.0, 500.0])
    assert abs(k2 - 1.0) < 1e-9, f"values proportional to n have slope +1, got {k2:.4f}"
    print("exercise 2 looks right — empirical_se uses n-1, and loglog_slope recovers an "
          "exact power law")

In [ ]:
_try("exercise 2", _check_se_and_slope)

## 4. Measuring the sampling distributions of a mean and a median

Now use them. For six sample sizes, simulate the sampling distribution of the **mean** and of
the **median**, estimate each one's standard error, and fit how it shrinks. Two more columns
say how *centred* and how *lopsided* the mean's distribution is: the average of the simulated
means, and the share of them that land below the true mean. For a symmetric sampling
distribution that share would be one half. Each sample size gets its own named stream, so no
two rows share draws. This cell uses your exercises 1 and 2.

In [ ]:
SE_NS = (10, 30, 100, 300, 1000, 3000)
SE_REPS = 2000


def show_sampling_distributions() -> None:
    rows = []
    for n in SE_NS:
        means = simulate_sampling_distribution(np.mean, n, SE_REPS, make_rng("se-mean", n))
        medians = simulate_sampling_distribution(np.median, n, SE_REPS, make_rng("se-median", n))
        rows.append((n, means, medians))
    print(f"{SE_REPS} simulated studies per row; truth: mean {TRUE_MEAN:.4f}, "
          f"median {TRUE_MEDIAN:.4f}\n")
    print(f"{'n':>6} {'mean of means':>14} {'SE(mean)':>9} {'sd/sqrt(n)':>11} {'ratio':>6} "
          f"{'below truth':>12} {'SE(median)':>11} {'SE(med)/SE(mean)':>17}")
    se_mean, se_median = [], []
    for n, means, medians in rows:
        a, b = empirical_se(means), empirical_se(medians)
        se_mean.append(a)
        se_median.append(b)
        theory = TRUE_SD / math.sqrt(n)
        print(f"{n:>6} {means.mean():>14.4f} {a:>9.4f} {theory:>11.4f} {a / theory:>6.2f} "
              f"{100 * np.mean(means < TRUE_MEAN):>11.1f}% {b:>11.4f} {b / a:>17.2f}")
    k_mean = loglog_slope(SE_NS, se_mean)
    k_median = loglog_slope(SE_NS, se_median)
    print(f"\nfitted log-log slope of the standard error against n: mean {k_mean:+.3f}, "
          f"median {k_median:+.3f}; with a finite variance the textbook says {-1 / 2:+.3f}.")
    ratios = [a / (TRUE_SD / math.sqrt(n)) for a, n in zip(se_mean, SE_NS)]
    low = int(np.argmin(ratios))
    print(f"the simulated SE of the mean is {100 * abs(1 - ratios[low]):.0f}% "
          f"{'below' if ratios[low] < 1 else 'above'} the formula at n = {SE_NS[low]} and "
          f"{100 * abs(1 - ratios[-1]):.0f}% {'below' if ratios[-1] < 1 else 'above'} it at "
          f"n = {SE_NS[-1]}.")
    if ratios[low] < 1:
        print("below, because much of this population's variance sits in giant draws so rare "
              f"that a typical run of {SE_REPS} simulated studies holds fewer of them than their "
              "long-run share (the rare run that catches one overshoots instead): a simulation "
              "can only show the tail it actually draws.")
    shares = [100 * np.mean(m < TRUE_MEAN) for _, m, _ in rows]
    print(f"{shares[0]:.1f}% of the simulated means at n = {SE_NS[0]} fall below the truth, "
          f"{shares[-1]:.1f}% at n = {SE_NS[-1]} (a symmetric distribution would give 50%).")
    if shares[0] > 55:
        print("so the sampling distribution of the mean is lopsided: a few big draws pull the "
              "mean up, and most studies land below the truth.")
    steadier = "median" if se_median[0] < se_mean[0] else "mean"
    print(f"the median's standard error is {se_median[0] / se_mean[0]:.2f} times the mean's at "
          f"n = {SE_NS[0]}: on this population the {steadier} is the steadier statistic, though "
          "the two estimate different things, so you cannot swap one for the other.")
    _RESULTS["se_slope_mean"] = k_mean
    _RESULTS["se_slope_median"] = k_median


_demo("sampling distributions demo", show_sampling_distributions,
      needs=("exercise 1", "exercise 2"))

## 5. Exercise 3 — the bootstrap, in two functions

You rarely know the population, so you cannot simulate from it. The **bootstrap** makes the
best available substitute: treat your own sample as if it were the population, and resample
from it *with replacement*, drawing as many observations as the sample has. Each resample is a
stand-in for "another study"; the statistic computed on each is a stand-in for one draw of the
sampling distribution. The spread of those stand-ins estimates the standard error, and their
**percentile interval** — the middle `level` of them — is a confidence interval.

Split the work in two. `bootstrap_distribution` produces the resampled statistics.
`percentile_interval` turns a pile of numbers into the interval whose ends leave
`(1 - level) / 2` of the numbers outside on each side. (`bootstrap_ci`, given below, composes
the two.) The method is Efron's (see `claims.yaml`).

<details><summary>💡 Hint 1 — what to think about</summary>

Resampling with replacement means every resample picks `n` row positions at random, positions
repeating freely — what happens to a statistic like the maximum if positions could NOT
repeat? And for the interval: for `level = 0.95`, how much probability is left over, and how
is it shared between the two ends?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Draw a matrix of random positions, `n_boot` rows by `n` columns, from `rng`; index the sample
with it so each row is one resample; give that matrix to the statistic, row-wise, once. For the
interval, split the leftover probability into two equal tails and ask numpy for the values at
those two cumulative probabilities.
</details>

In [ ]:
def bootstrap_distribution(sample: Sequence[float], statistic: Callable[..., np.ndarray],
                            n_boot: int, rng: np.random.Generator) -> np.ndarray:
    """The statistic on `n_boot` bootstrap resamples of `sample`.

    Requirements, each of which is graded:
      * each resample draws `len(sample)` values from `sample` WITH replacement, using the `rng`
        you were given.
      * `statistic` is called as `statistic(resamples, axis=1)` on the `(n_boot, n)` matrix of
        resamples and returns one value per row; return them as a float array of shape
        `(n_boot,)`.

    Example:
        >>> boots = bootstrap_distribution([1.0, 2.0, 3.0], np.mean, 4, make_rng("doc"))
        >>> boots.shape
        (4,)
        >>> bool(boots.min() >= 1.0 and boots.max() <= 3.0)   # a mean of copies of 1..3
        True
    """
    # YOUR CODE HERE
    raise NotImplementedError


def percentile_interval(boot_values: Sequence[float], level: float = LEVEL) -> tuple[float, float]:
    """The central `level` interval of `boot_values`: the values at cumulative probability
    `(1 - level) / 2` and `1 - (1 - level) / 2`, with numpy's default (linear) interpolation.

    Requirements, each of which is graded: the two ends leave equal probability outside on each
    side (so `level = 0.95` is the 2.5th and 97.5th percentiles); the input need not be sorted;
    `level = 1.0` returns the minimum and maximum, and `level = 0.0` the median twice.

    Example:
        >>> percentile_interval(list(range(101)), 0.90)
        (5.0, 95.0)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def bootstrap_ci(sample: Sequence[float], statistic: Callable[..., np.ndarray],
                  n_boot: int, level: float, rng: np.random.Generator) -> tuple[float, float]:
    """A percentile bootstrap confidence interval (given): `percentile_interval` of
    `bootstrap_distribution`."""
    return percentile_interval(bootstrap_distribution(sample, statistic, n_boot, rng), level)


# Public checks — run these as often as you like.
def _check_bootstrap() -> None:
    sample = np.arange(1.0, 11.0)
    boots = bootstrap_distribution(sample, np.max, 2000, make_rng("check", 6))
    assert boots.shape == (2000,), f"expected shape (2000,), got {boots.shape}"
    share_ten = float(np.mean(boots == 10.0))
    assert 0.55 < share_ten < 0.75, (
        f"the maximum of a with-replacement resample of 1..10 equals 10 about 65% of the time "
        f"(1 - 0.9**10); yours did {100 * share_ten:.0f}% — resample WITH replacement, so a "
        "position can repeat and the largest value can be missed"
    )

    def ones_per_row(a, axis):
        return np.sum(np.ones_like(a), axis=axis)

    sizes = bootstrap_distribution(sample, ones_per_row, 50, make_rng("check", 7))
    assert np.all(sizes == 10), (
        f"each resample must have as many values as the sample (10), got {np.unique(sizes)}"
    )
    again = bootstrap_distribution(sample, np.max, 2000, make_rng("check", 6))
    assert np.array_equal(boots, again), (
        "the same rng state must give the same resamples: take the random positions from `rng`"
    )
    lo, hi = percentile_interval(np.arange(1001.0), 0.95)
    assert abs(lo - 25.0) < 1e-6 and abs(hi - 975.0) < 1e-6, (
        f"the central 95% of 0..1000 runs from the 2.5th percentile (25) to the 97.5th (975), "
        f"got ({lo}, {hi}) — leave HALF of 1 - level in each tail"
    )
    lo9, hi9 = percentile_interval(np.arange(1001.0), 0.90)
    assert abs(lo9 - 50.0) < 1e-6 and abs(hi9 - 950.0) < 1e-6, (
        f"level 0.90 should give (50, 950), got ({lo9}, {hi9})"
    )
    print("exercise 3 looks right — resamples are with replacement and the same size as the "
          "sample, and the interval leaves equal tails")

In [ ]:
_try("exercise 3", _check_bootstrap)

## 6. One bootstrap, set beside the real sampling distribution

Take one sample of the size a small experiment might have, bootstrap it, and compare what the
bootstrap *estimates* with what the simulation of section 2 *measures* for the same `n`. The
bootstrap sees only your one sample; the simulation sees the whole population. This cell uses
your exercises 1 to 3.

In [ ]:
def show_one_bootstrap() -> None:
    n = 50
    sample = draw_population(make_rng("one-sample"), n)
    boots = bootstrap_distribution(sample, np.mean, 4000, make_rng("one-boot"))
    lo, hi = percentile_interval(boots, LEVEL)
    truth_se = empirical_se(simulate_sampling_distribution(np.mean, n, 4000, make_rng("one-truth")))
    print(f"one sample of n = {n}: mean {sample.mean():.4f} (true mean {TRUE_MEAN:.4f})")
    print(f"bootstrap SE of the mean, from this one sample: {empirical_se(boots):.4f}")
    print(f"simulated SE of the mean at n = {n}, from the population:  {truth_se:.4f}")
    print(f"{100 * LEVEL:.0f}% percentile interval: [{lo:.4f}, {hi:.4f}], width {hi - lo:.4f}; "
          f"it {'covers' if lo <= TRUE_MEAN <= hi else 'misses'} the true mean {TRUE_MEAN:.4f}.")
    boot_se = empirical_se(boots)
    print(f"this sample's bootstrap SE is {100 * abs(1 - boot_se / truth_se):.0f}% "
          f"{'below' if boot_se < truth_se else 'above'} the simulated one. A bootstrap SE comes "
          "from ONE sample, so it is itself random: a sample that happens to contain no giant "
          "draws gives a small bootstrap SE, because the bootstrap can only resample what the "
          "sample holds. The next exercise measures how often that turns into a wrong interval.")


_demo("one bootstrap demo", show_one_bootstrap,
      needs=("exercise 1", "exercise 2", "exercise 3"))

## 7. Exercise 4 — does a "95% interval" cover 95% of the time?

"95%" is not a property of the interval you computed; it is a claim about the *procedure*: run
it on many fresh samples and the interval should contain the truth in 95% of them. Because we
know the truth, we can count. `coverage_rate` does the counting: for each of `trials` trials
draw a fresh sample of size `n`, build an interval from it with the function you are given, and
record whether the interval contains `true_value`. It also reports the **Monte-Carlo standard
error** of the rate it measured, so you know how many points of a gap are noise.

<details><summary>💡 Hint 1 — what to think about</summary>

Is an interval whose end sits exactly on the truth covering it? Decide, then keep that rule for
both ends. And how precisely can a fraction measured from `trials` yes/no outcomes be known?
It is a proportion: its standard error comes from the proportion itself and `trials`.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Loop `trials` times: draw a fresh sample of `n` from the population using `rng`, call
`interval_fn(sample, rng)`, and add one to a counter when the lower end is at most the truth and
the upper end at least the truth. The rate is the counter over `trials`; its Monte-Carlo
standard error is the square root of rate times (1 - rate) over `trials`.
</details>

In [ ]:
def coverage_rate(interval_fn: Callable[[np.ndarray, np.random.Generator], tuple[float, float]],
                   true_value: float, n: int, trials: int,
                   rng: np.random.Generator) -> tuple[float, float]:
    """The fraction of `trials` fresh samples whose interval contains `true_value`, and that
    fraction's Monte-Carlo standard error.

    Requirements, each of which is graded:
      * each trial draws a FRESH sample of `n` values from `draw_population(rng, n)`, then calls
        `interval_fn(sample, rng)`, which returns `(lo, hi)`.
      * the interval covers the truth when `lo <= true_value <= hi`: both ends are inclusive.
      * returns `(rate, mc_se)`: `rate = covered / trials` as a fraction between 0 and 1, and
        `mc_se = sqrt(rate * (1 - rate) / trials)`.

    Example:
        >>> rate, se = coverage_rate(lambda s, r: (0.0, 100.0), TRUE_MEAN, 10, 5, make_rng("doc"))
        >>> (rate, se)
        (1.0, 0.0)
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_coverage_rate() -> None:
    seen: list[np.ndarray] = []

    def spy(sample, rng):
        seen.append(np.asarray(sample))
        return (TRUE_MEAN - 1.0, TRUE_MEAN + 1.0)

    rate, se = coverage_rate(spy, TRUE_MEAN, 7, 20, make_rng("check", 8))
    assert len(seen) == 20, f"interval_fn must be called once per trial (20), got {len(seen)}"
    assert all(len(s) == 7 for s in seen), "every trial's sample must have exactly n = 7 draws"
    assert len({tuple(s.tolist()) for s in seen}) == 20, "every trial needs a FRESH sample"
    assert rate == 1.0 and se == 0.0, f"an interval that always covers gives (1.0, 0.0), got ({rate}, {se})"
    r0, _ = coverage_rate(lambda s, r: (TRUE_MEAN + 1.0, TRUE_MEAN + 2.0), TRUE_MEAN, 7, 20,
                          make_rng("check", 9))
    assert r0 == 0.0, f"an interval that never covers gives 0.0, got {r0}"
    r_edge, _ = coverage_rate(lambda s, r: (TRUE_MEAN, TRUE_MEAN + 1.0), TRUE_MEAN, 7, 20,
                              make_rng("check", 10))
    assert r_edge == 1.0, (
        "an interval whose lower end is exactly the truth still contains it: use <=, not <"
    )
    print("exercise 4 looks right — fresh samples, inclusive ends, a fraction between 0 and 1 "
          "and its Monte-Carlo standard error")

In [ ]:
_try("exercise 4", _check_coverage_rate)

## 8. How often are bootstrap intervals right?

Now measure it: the percentile bootstrap interval for the **mean** and for the **median**, at
three sample sizes, each `COVER_TRIALS` times. The nominal coverage is `LEVEL`. A row is flagged
only when its coverage differs from nominal by more than two Monte-Carlo standard errors, so
a gap that is just noise is not called a finding. This cell uses your exercises 3 and 4 and
takes a few seconds.

In [ ]:
COVER_NS = (20, 100, 400)
COVER_TRIALS = 500
COVER_BOOT = 1000


def show_bootstrap_coverage() -> None:
    print(f"{COVER_TRIALS} fresh samples per row, {COVER_BOOT} bootstrap resamples each; "
          f"nominal coverage {100 * LEVEL:.0f}%\n")
    print(f"{'statistic':<10} {'n':>5} {'coverage':>9} {'MC SE':>7} {'gap':>8}   verdict")
    worst: dict[str, tuple[float, int]] = {}
    borderline = 0
    for name, stat, truth in (("mean", np.mean, TRUE_MEAN), ("median", np.median, TRUE_MEDIAN)):
        for n in COVER_NS:
            def interval_fn(sample, rng, stat=stat):
                return bootstrap_ci(sample, stat, COVER_BOOT, LEVEL, rng)
            rate, se = coverage_rate(interval_fn, truth, n, COVER_TRIALS, make_rng("cover", name, n))
            gap = 100 * (rate - LEVEL)
            if se > 0 and abs(rate - LEVEL) > 2 * se:
                verdict = "under-covers" if rate < LEVEL else "over-covers"
            else:
                verdict = "consistent with nominal"
            if se > 0 and 1 <= abs(rate - LEVEL) / se <= 3:
                verdict += " (borderline)"
                borderline += 1
            print(f"{name:<10} {n:>5} {100 * rate:>8.1f}% {100 * se:>6.1f}% {gap:>+7.1f}   {verdict}")
            if name not in worst or rate < worst[name][0]:
                worst[name] = (rate, n)
    if borderline:
        print(f"{borderline} row(s) marked borderline sit within one Monte-Carlo SE of the two-SE "
              "line, so a different seed could print the other verdict word: read those rows by "
              "their gap and MC SE, not by the word.")
    for name, (rate, n) in worst.items():
        print(f"lowest coverage for the {name}: {100 * rate:.1f}% at n = {n}")
    _RESULTS["cover_mean_lowest"] = worst["mean"][0]
    _RESULTS["cover_median_lowest"] = worst["median"][0]
    if any(r < LEVEL for r, _ in worst.values()):
        print(f"a '{100 * LEVEL:.0f}%' interval that covers the truth less often than that is "
              "overconfident: it reports more certainty than the data supports. Compare the "
              "rows by statistic and by n to see where the shortfall is largest.")


_demo("bootstrap coverage demo", show_bootstrap_coverage,
      needs=("exercise 3", "exercise 4"))

## 9. Exercise 5 — a permutation test, with no formula in sight

You compared two groups and their means differ by `d_obs`. Is that more than chance? Ask the
question literally. **If the group labels meant nothing** — if both groups came from the same
population — then any way of splitting the pooled observations into groups of the same sizes
was equally likely to have been the one you saw. So shuffle the labels many times, recompute
the difference each time, and count how often a shuffle produces a difference at least as
extreme as the one you observed. That fraction is the **p-value**: no normal curve, no
variance formula, no assumption about the tail.

Three details decide whether your p-value is honest, and each is graded:

* **Count ties.** A shuffle that gives *exactly* your observed difference is as extreme as it,
  so it counts: compare with "at least as extreme", not "more extreme".
* **Mind the last bit.** Two shuffles that are the same in arithmetic can differ in the last
  binary digit of a float, and "exactly equal" then fails. Count a shuffle when it comes within
  `TIE_TOL` of the observed value on the extreme side.
* **Count your own data too.** The labels you observed are one of the possible arrangements, so
  `p = (1 + count) / (n_perm + 1)`. This also means a p-value can never be exactly 0, which
  would claim an impossibility. (Phipson and Smyth give the argument; see `claims.yaml`. The
  rule is conservative when shuffles are drawn with replacement, as yours are.)

<details><summary>💡 Hint 1 — what to think about</summary>

Pool the two samples into one array and remember how many belong to `x`. A shuffle is a random
ordering of the pooled array: the first `len(x)` positions are the new "x", the rest the new
"y". For "two-sided", which direction of difference counts as extreme? Both, by size.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Many random orderings at once: draw a matrix of uniform random numbers, one row per shuffle,
and sort each row to get the ordering (a row-wise argsort of random numbers is a random
permutation). Index the pooled array with that matrix, split each row at `len(x)`, take the
two row means and subtract. Then count rows by the rule for the chosen alternative, using
`TIE_TOL`, and apply the add-one rule.
</details>

In [ ]:
TIE_TOL = 1e-12     # how close counts as "equal" when comparing two floating-point differences.


def permutation_test(x: Sequence[float], y: Sequence[float], n_perm: int,
                      rng: np.random.Generator, alternative: str = "two-sided") -> float:
    """The permutation p-value for a difference in means, `mean(x) - mean(y)`.

    Requirements, each of which is graded:
      * `n_perm` random re-assignments of the pooled values to a group of `len(x)` and a group of
        `len(y)`, drawn from `rng`; the statistic of each is its `mean(x') - mean(y')`.
      * "extreme" with `d` a shuffled difference and `obs` the observed one:
        `"two-sided"`: `abs(d) >= abs(obs) - TIE_TOL`; `"greater"`: `d >= obs - TIE_TOL`;
        `"less"`: `d <= obs + TIE_TOL`. Any other `alternative` raises `ValueError`.
      * `p = (1 + number of extreme shuffles) / (n_perm + 1)`.
      * the inputs `x` and `y` are not modified.

    Example:
        >>> p = permutation_test([1.0, 2.0], [3.0, 4.0], 2000, make_rng("doc"))
        >>> 0.2 < p < 0.5      # 2 of the 6 possible splits are as extreme: about 1/3
        True
    """
    # YOUR CODE HERE
    raise NotImplementedError


def exact_permutation_p(x: Sequence[float], y: Sequence[float]) -> float:
    """The EXACT two-sided permutation p-value, by listing every way to choose which pooled
    values form the first group (given; only possible for a handful of values). It is what a
    Monte-Carlo `permutation_test` is trying to estimate."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    pooled = np.concatenate([x, y])
    observed = abs(x.mean() - y.mean())
    count = total = 0
    for chosen in itertools.combinations(range(pooled.size), x.size):
        mask = np.zeros(pooled.size, dtype=bool)
        mask[list(chosen)] = True
        count += abs(pooled[mask].mean() - pooled[~mask].mean()) >= observed - TIE_TOL
        total += 1
    return count / total


# Public checks — run these as often as you like.
def _check_permutation_test() -> None:
    x, y = [1.0, 2.0], [3.0, 4.0]
    p = permutation_test(x, y, 4000, make_rng("check", 11))
    assert abs(p - exact_permutation_p(x, y)) < 0.04, (
        f"x = [1, 2] against y = [3, 4]: of the 6 equally likely splits, 2 are as extreme in "
        f"either direction, so p is about 1/3; yours was {p:.3f}. Count BOTH directions for "
        "'two-sided', and count a tie as extreme (>=, not >)"
    )
    p_less = permutation_test(x, y, 4000, make_rng("check", 12), alternative="less")
    assert abs(p_less - 1 / 6) < 0.04, (
        f"x is below y, so alternative='less' sees 1 of 6 splits as extreme (about 0.17); "
        f"yours was {p_less:.3f}"
    )
    apart_x, apart_y = list(range(1, 11)), list(range(101, 111))
    p_floor = permutation_test(apart_x, apart_y, 19, make_rng("check", 13), alternative="less")
    assert abs(p_floor - 1 / 20) < 1e-12, (
        f"with 19 shuffles and no shuffle as extreme as completely separated groups, the p-value "
        f"is (1 + 0) / (19 + 1) = 0.05, never 0; yours was {p_floor}"
    )
    keep_x = np.array(apart_x, dtype=float)
    permutation_test(keep_x, apart_y, 19, make_rng("check", 14))
    assert np.array_equal(keep_x, np.array(apart_x, dtype=float)), (
        "permutation_test must not shuffle or overwrite the arrays it was given"
    )
    print("exercise 5 looks right — ties count, the p-value is (1 + count) / (n_perm + 1), and "
          "the one-sided and two-sided versions agree with the exact enumeration")

In [ ]:
_try("exercise 5", _check_permutation_test)

## 10. Is the permutation p-value calibrated, and what does the formula do?

A p-value is calibrated if, when there is NO real difference, it falls below 0.05 in about 5%
of experiments. We can test that. Simulate many experiments in which both groups come from the
SAME heavy-tailed population, run your permutation test on each, and count. Put the textbook
normal-approximation test (the Welch z, with the given `welch_z`) through the same experiments
for comparison, then through ten times as many on its own, because a formula is cheap. A
Monte-Carlo standard error says how much of any gap is noise. First, a tiny
case small enough to check against exact enumeration. This cell uses your exercise 5.

In [ ]:
NULL_SIMS = 3000
NULL_N = 30
NULL_PERM = 500
NULL_Z_SIMS = 30000       # the Welch z alone, on many more experiments: it costs almost nothing


def show_permutation_calibration() -> None:
    x, y = [1.1, 2.2, 3.3], [4.4, 5.5, 6.6]
    exact = exact_permutation_p(x, y)
    est = permutation_test(x, y, 20000, make_rng("perm-tiny"))
    print(f"tiny case x = {x}, y = {y}: exact p {exact:.4f} (listing all splits), "
          f"permutation_test with 20000 shuffles {est:.4f}")
    print("(a computer stores decimals such as 1.1 inexactly, so two shuffles that are equal in "
          "arithmetic can differ in the last bit: that is why TIE_TOL exists.)\n")

    rng = make_rng("perm-null")
    perm_p, z = [], []
    for _ in range(NULL_SIMS):
        a = draw_population(rng, NULL_N)
        b = draw_population(rng, NULL_N)
        perm_p.append(permutation_test(a, b, NULL_PERM, rng))
        z.append(float(welch_z(a, b)))
    perm_p = np.array(perm_p)
    z_p = two_sided_p(np.array(z))
    print(f"{NULL_SIMS} experiments with NO real difference, {NULL_N} users per group:")
    print(f"{'test':<28} {'rejects at 5%':>14} {'MC SE':>7}   verdict")
    for name, p in (("permutation test", perm_p), ("Welch z, normal formula", z_p)):
        rate = float(np.mean(p <= ALPHA))
        se = math.sqrt(ALPHA * (1 - ALPHA) / NULL_SIMS)
        verdict = ("consistent with nominal" if abs(rate - ALPHA) <= 2 * se
                   else "rejects too often" if rate > ALPHA else "rejects too rarely")
        if 1 <= abs(rate - ALPHA) / se <= 3:
            verdict += " (borderline: another seed could flip this word)"
        print(f"{name:<28} {100 * rate:>13.1f}% {100 * se:>6.1f}%   {verdict}")
    # The formula is cheap, so measure its own rate far more precisely on many more experiments.
    rng_z = make_rng("perm-null", "z-only")
    big_z = welch_z(draw_population(rng_z, (NULL_Z_SIMS, NULL_N)),
                    draw_population(rng_z, (NULL_Z_SIMS, NULL_N)))
    z_rate = float(np.mean(two_sided_p(big_z) <= ALPHA))
    z_se = math.sqrt(ALPHA * (1 - ALPHA) / NULL_Z_SIMS)
    print(f"the Welch z alone on {NULL_Z_SIMS} further experiments (a formula is cheap): rejects in "
          f"{100 * z_rate:.1f}% (MC SE {100 * z_se:.1f}%), {abs(z_rate - ALPHA) / z_se:.1f} "
          "standard errors from the nominal 5%.")
    if abs(z_rate - ALPHA) > 2 * z_se:
        print(f"so at {NULL_N} users per group the normal formula is "
              + ("conservative: it rejects less often" if z_rate < ALPHA else
                 "anti-conservative: it rejects more often")
              + f" than the {100 * ALPHA:.0f}% it promises. A heavy-tailed mean is not yet normal, "
              "so the formula's p-values are only approximately right. The permutation test makes "
              "no such assumption.")
    else:
        print(f"so at {NULL_N} users per group the normal formula's rate cannot be told from the "
              f"nominal {100 * ALPHA:.0f}% at this precision.")
    print(f"share of permutation p-values at or below 0.5: {100 * np.mean(perm_p <= 0.5):.1f}% "
          "(a calibrated p-value is uniform when there is no effect, so about half)")
    print(f"smallest permutation p-value in the {NULL_SIMS} experiments: {perm_p.min():.4f}; "
          f"the floor with {NULL_PERM} shuffles is {1 / (NULL_PERM + 1):.4f}")
    _RESULTS["null_perm_rate"] = float(np.mean(perm_p <= ALPHA))
    _RESULTS["null_z_rate"] = z_rate


_demo("permutation calibration demo", show_permutation_calibration, needs=("exercise 5",))

## 11. Exercise 6 — power, by simulation

The p-value answers "is this difference more than chance?". **Power** answers the planning
question before you spend anything: *if the treatment really does lift the mean by a given relative amount, how
often will my experiment detect it?* There is no formula to look up. Simulate the experiment
many times with the lift built in, run the test each time, and count how often it rejects.

`estimate_power` runs that simulation for a test you pass in as a function `test(x, y, rng)`
that returns a p-value — so the same machinery measures your permutation test, a z-test, or
anything else. The control group is the population; the treatment group is the population
**multiplied** by `1 + lift`, a *relative* lift, the way product teams state them. The test
rejects when `p <= alpha`. Like `coverage_rate`, it also returns a Monte-Carlo standard error.

<details><summary>💡 Hint 1 — what to think about</summary>

Which group is multiplied, and by what? A relative lift is a percentage of what the same
user would have produced under control: which factor turns a control value into a treatment
one? What happens when `lift` is 0? And a p-value that lands exactly on `alpha`: reject or
not? The statement above says which.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Loop `n_sims` times: draw a control sample and a second sample of the same size, scale the
second by one plus the lift, call the test on `(x, y, rng)` and count the p-values at or below
`alpha`. Power is that count over `n_sims`; the Monte-Carlo standard error has the same form
as in exercise 4.
</details>

In [ ]:
def estimate_power(n_per_group: int, lift: float,
                    test: Callable[[np.ndarray, np.ndarray, np.random.Generator], float],
                    n_sims: int, alpha: float, rng: np.random.Generator) -> tuple[float, float]:
    """The fraction of `n_sims` simulated experiments in which `test` rejects, and that
    fraction's Monte-Carlo standard error.

    Requirements, each of which is graded:
      * per simulation: `x = draw_population(rng, n_per_group)` is the control group;
        `y = (1 + lift) * draw_population(rng, n_per_group)` is the treatment group (a RELATIVE
        lift: the whole distribution is scaled); the p-value is `test(x, y, rng)`.
      * the test rejects when `p <= alpha` (a p-value exactly equal to `alpha` rejects).
      * returns `(power, mc_se)` with `mc_se = sqrt(power * (1 - power) / n_sims)`.

    Example:
        >>> power, se = estimate_power(10, 0.3, lambda x, y, r: 0.01, 8, 0.05, make_rng("doc"))
        >>> power
        1.0
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_estimate_power() -> None:
    calls = []

    def spy(x, y, rng):
        calls.append((len(x), len(y)))
        return 0.05

    power, se = estimate_power(12, 0.3, spy, 9, 0.05, make_rng("check", 15))
    assert len(calls) == 9 and all(c == (12, 12) for c in calls), (
        "call the test once per simulation, with two samples of n_per_group each"
    )
    assert power == 1.0 and se == 0.0, (
        f"a p-value exactly equal to alpha rejects (p <= alpha), so a test that always returns "
        f"0.05 at alpha = 0.05 has power 1.0; got ({power}, {se})"
    )
    never, _ = estimate_power(12, 0.3, lambda x, y, r: np.nextafter(0.05, 1.0), 9, 0.05,
                              make_rng("check", 16))
    assert never == 0.0, f"a p-value just above alpha does not reject, got power {never}"
    ratios = []
    estimate_power(3000, 1.0, lambda x, y, r: ratios.append(np.median(y) / np.median(x)) or 1.0,
                   1, 0.05, make_rng("check", 17))
    assert 1.8 < ratios[0] < 2.2, (
        f"a lift of 1.0 doubles the treatment group: its median should be about twice the "
        f"control's, got a ratio of {ratios[0]:.2f} — multiply by (1 + lift), do not add"
    )
    print("exercise 6 looks right — relative lift on the treatment group, rejection at p <= alpha, "
          "and a Monte-Carlo standard error")

In [ ]:
_try("exercise 6", _check_estimate_power)

## 12. Exercise 7 — the smallest sample that reaches the power you want

A power calculation ends in a decision: *how many users per group do we need?* You simulate
power at a grid of sample sizes and pick the smallest that reaches your target (80% is the usual
one). Because each power is itself a noisy simulation, the curve can wiggle: the rule here is
the plain one a planner uses. Scan the sizes in the order given, and return the **first** one
whose power is at least the target; a power that exactly equals the target counts.

<details><summary>💡 Hint — the approach, in words</summary>

Walk the two lists together, in order, and stop at the first size that qualifies. What should
come back if none does? The statement says: nothing at all, which in Python is `None`, so a
caller can tell "no size in this grid is enough" from a real answer.
</details>

In [ ]:
def smallest_n_for_power(ns: Sequence[int], powers: Sequence[float], target: float):
    """The first sample size in `ns` (scanned in the given order) whose power in `powers` is at
    least `target`, or `None` if no size reaches it.

    Requirements, each of which is graded: a power exactly equal to `target` qualifies; the FIRST
    qualifying size in the given order is returned even if a later, larger size dips back below
    the target (power estimates are noisy); the grid is walked in the order given, sorted or
    not; the size itself is returned, not its position.

    Example:
        >>> smallest_n_for_power([25, 50, 100], [0.41, 0.77, 0.93], 0.8)
        100
        >>> smallest_n_for_power([25, 50], [0.41, 0.77], 0.8) is None
        True
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_smallest_n() -> None:
    assert smallest_n_for_power([10, 20, 30, 40], [0.5, 0.7, 0.85, 0.95], 0.8) == 30, (
        "30 is the first size whose power (0.85) reaches 0.8"
    )
    assert smallest_n_for_power([10, 20, 30], [0.5, 0.8, 0.9], 0.8) == 20, (
        "a power exactly equal to the target qualifies: use >=, not >"
    )
    assert smallest_n_for_power([10, 20], [0.2, 0.3], 0.8) is None, (
        "when no size reaches the target, return None, not the largest size"
    )
    print("exercise 7 looks right — the first size to reach the target, or None")

In [ ]:
_try("exercise 7", _check_smallest_n)

## 13. A power curve for a relative lift

Put the pieces together. For a grid of sample sizes, estimate the power of your permutation
test against a lift of `LIFT`, and let `smallest_n_for_power` pick a size for `TARGET_POWER`.
A control row with a lift of zero checks the machinery: with no effect, "power" is just the
false-positive rate and should sit near `ALPHA`. Each power carries its Monte-Carlo standard
error, so read the table as a range, not a point. This cell uses exercises 5 to 7.

In [ ]:
LIFT = 0.30
TARGET_POWER = 0.80
POWER_NS = (25, 50, 100, 150, 200, 300)
POWER_SIMS = 400
POWER_PERM = 199


def show_power_curve() -> None:
    def perm(x, y, rng):
        return permutation_test(x, y, POWER_PERM, rng)

    print(f"{POWER_SIMS} simulated experiments per row, permutation test with {POWER_PERM} "
          f"shuffles, alpha {ALPHA:g}\n")
    print(f"{'users per group':>16} {'power':>7} {'MC SE':>7}")
    powers, ses = [], []
    for n in POWER_NS:
        p, se = estimate_power(n, LIFT, perm, POWER_SIMS, ALPHA, make_rng("power", n))
        powers.append(p)
        ses.append(se)
        print(f"{n:>16} {100 * p:>6.1f}% {100 * se:>6.1f}%")
    chosen = smallest_n_for_power(POWER_NS, powers, TARGET_POWER)
    null_power, null_se = estimate_power(POWER_NS[2], 0.0, perm, POWER_SIMS, ALPHA,
                                         make_rng("power-null"))
    print(f"\ncontrol, lift 0 at {POWER_NS[2]} users per group: rejects in {100 * null_power:.1f}% "
          f"(MC SE {100 * null_se:.1f}%) of experiments, against a nominal {100 * ALPHA:.0f}%")
    if chosen is None:
        print(f"no size on this grid reaches {100 * TARGET_POWER:.0f}% power for a "
              f"{100 * LIFT:.0f}% lift: the grid stops at {POWER_NS[-1]} users per group.")
    else:
        print(f"smallest size on the grid reaching {100 * TARGET_POWER:.0f}% power for a "
              f"{100 * LIFT:.0f}% lift: {chosen} users per group.")
        i = POWER_NS.index(chosen)
        if i > 0:
            gap = TARGET_POWER - powers[i - 1]
            print(f"the size before it, {POWER_NS[i - 1]}, measured {100 * powers[i - 1]:.1f}%: "
                  f"{100 * gap:.1f} points short, which is {gap / ses[i - 1]:.1f} Monte-Carlo "
                  "standard errors. " + ("Within two of them, that miss cannot be told from "
                  "noise." if gap <= 2 * ses[i - 1] else "Well outside noise: it really falls short."))
        print(f"a planner who needs the target to hold rounds up, or simulates more: every "
              "power here is an estimate with its own error.")
    _RESULTS["power_null"] = null_power


_demo("power curve demo", show_power_curve,
      needs=("exercise 5", "exercise 6", "exercise 7"))

## 14. Peeking: what a running test does to your false-positive rate

The p-value's 5% promise is for ONE look, at a sample size fixed in advance. A dashboard that
updates every day invites a different procedure: look, and if `p < 0.05`, stop and ship; if
not, wait for more data and look again. Each look is another chance for noise to cross the
line, and the chances add up (Johari and colleagues show the plain p-value is unreliable
under continuous monitoring; see `claims.yaml`). We can measure how much here. Simulate
experiments in which both groups come from the SAME population (so every "significant"
result is a false positive), watch them grow batch by batch, and compute the test statistic
after every batch.

Two exercises. `z_statistics_at_looks` computes, for every simulated experiment, the Welch z
statistic using only the first `look` users per group, for each look in a list: row `i`,
column `j` is experiment `i` as seen at the `j`-th look. `rejection_rates` then asks the two
questions that matter: how often is the test significant at the FINAL look alone (the
experiment you planned), and how often is it significant at ANY look (the experiment you
actually ran when you stopped at the first one)?

<details><summary>💡 Hint 1 — what to think about</summary>

"Using only the first `look` users" means the first `look` columns of each row, not a random
subset and not the last ones: data arrives in order. The z formula is the one in the given
`welch_z`, with the `n - 1` variance. For the rates: a z of either sign is evidence, and a z
exactly at the critical value rejects (the same rule as `p <= alpha`).
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

For each look, slice both matrices to their first `look` columns and compute the statistic
along each row; stack the results as columns, in the order the looks were given. For the
rates, mark every entry whose absolute value reaches the critical value; the final-look rate
is the average of the last column, and the any-look rate is the average, over rows, of
"at least one marked entry in this row".
</details>

In [ ]:
def z_statistics_at_looks(x: np.ndarray, y: np.ndarray, looks: Sequence[int]) -> np.ndarray:
    """The Welch z statistic of every experiment at every look.

    `x` and `y` are `(sims, n_max)` matrices: row `i` is experiment `i`, its users in the order
    they arrived. Returns an array of shape `(sims, len(looks))` whose column `j` uses only the
    first `looks[j]` users of each group in each row.

    Requirements, each of which is graded: the statistic is
    `(mean x - mean y) / sqrt(var x / look + var y / look)` over those first `look` users, with
    the `n - 1` variance (the given `welch_z` does exactly this on one slice); the columns follow
    the order of `looks`; every look is at least 2 and at most `n_max`.

    Example:
        >>> x = np.array([[1.0, 3.0, 9.0]]); y = np.array([[0.0, 0.5, 9.0]])
        >>> z_statistics_at_looks(x, y, [2]).shape
        (1, 1)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def rejection_rates(z: np.ndarray, crit: float) -> tuple[float, float]:
    """`(final_rate, any_rate)` for a matrix `z` of z statistics, one row per experiment and one
    column per look: the fraction of experiments significant at the LAST look, and the fraction
    significant at AT LEAST ONE look.

    Requirements, each of which is graded: an entry is significant when `abs(z) >= crit` (either
    sign; equal to `crit` counts); both results are fractions between 0 and 1 as plain floats.

    Example:
        >>> rejection_rates(np.array([[2.5, 0.1], [0.1, 0.1]]), 2.0)
        (0.0, 0.5)
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_z_at_looks() -> None:
    x = np.array([[1.0, 3.0, 99.0]])
    y = np.array([[0.0, 0.5, 99.0]])
    z = z_statistics_at_looks(x, y, [2])
    assert z.shape == (1, 1), f"expected shape (sims, looks) = (1, 1), got {z.shape}"
    expected = (2.0 - 0.25) / math.sqrt(2.0 / 2 + 0.125 / 2)
    assert abs(z[0, 0] - expected) < 1e-9, (
        f"first two users per group: means 2 and 0.25, variances (n-1) 2 and 0.125, so z = "
        f"{expected:.4f}; got {z[0, 0]:.4f} — use only the FIRST `look` users, with ddof=1"
    )
    wide = z_statistics_at_looks(np.tile(x, (4, 1)), np.tile(y, (4, 1)), [3, 2])
    assert wide.shape == (4, 2) and abs(wide[0, 1] - expected) < 1e-9, (
        "columns must follow the order of `looks` (here 3 then 2)"
    )
    print("exercise 8 looks right — a Welch z per experiment per look, from the first `look` "
          "users only")


def _check_rejection_rates() -> None:
    grid = np.array([[2.5, 0.1], [0.1, 2.6], [0.1, -2.7], [0.1, 0.1], [-3.0, -3.0]])
    final_rate, any_rate = rejection_rates(grid, 2.0)
    assert math.isclose(final_rate, 0.6, abs_tol=1e-12) and math.isclose(any_rate, 0.8, abs_tol=1e-12), (
        f"3 of 5 experiments are significant at the last look and 4 of 5 at some look; "
        f"got ({final_rate}, {any_rate}) — a negative z counts too, so compare abs(z)"
    )
    edge, _ = rejection_rates(np.array([[1.96]]), 1.96)
    assert math.isclose(edge, 1.0, abs_tol=1e-12), "a z exactly at the critical value rejects: use >=, not >"
    print("exercise 9 looks right — the final-look rate and the any-look rate")

In [ ]:
_try("exercise 8", _check_z_at_looks)
_try("exercise 9", _check_rejection_rates)

## 15. Measuring the damage

`PEEK_SIMS` null experiments of `PEEK_NMAX` users per group, with `PEEK_LOOKS` equally spaced
looks. For each number of looks `K` (taking the first look at one `K`-th of the data, and so
on up to the final one) measure: the rate when you look ONCE at the end, the rate when you stop
at the first significant look, and the rate when you do look `K` times but apply a Bonferroni
correction to the threshold (compare against the level `ALPHA / K` instead of `ALPHA`). All
the "significant" results here are false positives, because both groups are identical. This
cell uses exercises 8 and 9.

In [ ]:
PEEK_SIMS = 10000
PEEK_NMAX = 400
PEEK_LOOKS = tuple(range(25, PEEK_NMAX + 1, 25))      # 16 looks, every 25 users per group
PEEK_KS = (1, 2, 4, 8, 16)


def show_peeking() -> None:
    rng = make_rng("peek")
    x = draw_population(rng, (PEEK_SIMS, PEEK_NMAX))
    y = draw_population(rng, (PEEK_SIMS, PEEK_NMAX))
    z = z_statistics_at_looks(x, y, PEEK_LOOKS)
    crit = normal_critical(ALPHA)
    n_looks = len(PEEK_LOOKS)
    print(f"{PEEK_SIMS} experiments with no real difference, up to {PEEK_NMAX} users per group, "
          f"critical |z| {crit:.3f} for a nominal {100 * ALPHA:.0f}%\n")
    print(f"{'looks':>6} {'one look at the end':>20} {'stop at first p<0.05':>21} "
          f"{'inflation':>10} {'Bonferroni over looks':>22}")
    rows = []
    for k in PEEK_KS:
        step = n_looks // k
        sub = z[:, step - 1::step]
        final_rate, any_rate = rejection_rates(sub, crit)
        _, bonf_rate = rejection_rates(sub, normal_critical(ALPHA / k))
        rows.append((k, final_rate, any_rate, bonf_rate))
        print(f"{k:>6} {100 * final_rate:>19.1f}% {100 * any_rate:>20.1f}% "
              f"{any_rate / final_rate:>9.1f}x {100 * bonf_rate:>21.1f}%")
    k, final_rate, any_rate, bonf_rate = rows[-1]
    se = math.sqrt(final_rate * (1 - final_rate) / PEEK_SIMS)
    nominal_se = math.sqrt(ALPHA * (1 - ALPHA) / PEEK_SIMS)
    away = (final_rate - ALPHA) / se
    print(f"\nlooking once, the false-positive rate is {100 * final_rate:.1f}% (Monte-Carlo SE "
          f"{100 * se:.1f}%), {abs(away):.1f} standard errors "
          + ("below" if away < 0 else "above") + f" the nominal {100 * ALPHA:.0f}%"
          + (": the normal formula is itself slightly conservative here, which is why the "
             "inflation column divides by this measured one-look rate, not by 5%."
             if away < -2 else "." if abs(away) <= 2 else
             ": the normal formula itself rejects too often here, before any peeking."))
    print(f"stopping at the first significant of {k} looks it is {100 * any_rate:.1f}%: "
          f"{any_rate / final_rate:.1f} times that one-look rate and {any_rate / ALPHA:.1f} times "
          f"the nominal {100 * ALPHA:.0f}%.")
    if bonf_rate <= ALPHA + 2 * nominal_se:
        print(f"dividing the level by the number of looks brings that to {100 * bonf_rate:.1f}%, "
              "so the repair holds"
              + (", with room to spare: the corrected test is conservative, a price paid in power."
                 if bonf_rate < ALPHA - 2 * nominal_se else "."))
    else:
        print(f"dividing the level by the number of looks brings that to {100 * bonf_rate:.1f}%, "
              f"still above the nominal {100 * ALPHA:.0f}%: the repair did not hold here.")
    _RESULTS["peek_final"] = final_rate
    _RESULTS["peek_any"] = any_rate
    _RESULTS["peek_k"] = float(k)


_demo("peeking demo", show_peeking, needs=("exercise 8", "exercise 9"))

## 16. Exercise 10 — twenty metrics at once, and two ways to pay for it

A dashboard shows twenty metrics for the experiment. If every one is tested at 5% and none of
them is truly affected, the chance that at least one lights up is not 5%: each metric is a
separate 5% chance. A **correction** raises the bar so the chance of *any* false alarm across
the whole family is held at `alpha`.

* **Bonferroni** compares each p-value to `alpha / m`, with `m` the number of tests. Simple
  and safe, and costly: it throws away power on every metric.
* **Holm** (step-down) never rejects less than Bonferroni and sometimes rejects more. Sort the p-values from smallest to
  largest; compare the smallest to `alpha / m`, the next to `alpha / (m - 1)`, and so on, up
  to the largest against `alpha`. Reject while each comparison passes, and **stop at the first
  one that fails**: that metric and every larger p-value stay unrejected, even if a later
  comparison would have passed on its own.

Both return a boolean mask in the ORIGINAL order of the p-values.

<details><summary>💡 Hint 1 — what to think about</summary>

Holm walks the sorted order, but the answer must come back in the order the p-values were
given: keep the argsort to map back. Which comparison does the stopping rule need — the rank
of the p-value in the sorted order, or its position in the original array?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Bonferroni is one vector comparison. For Holm: sort the indices by p-value; go through them in
that order with a counter starting at 0, where the threshold for step `i` is `alpha` divided by
the number of tests still left (`m - i`); mark each index rejected while the comparison holds
(at or below), and `break` out of the loop at the first failure.
</details>

In [ ]:
def bonferroni_reject(pvals: Sequence[float], alpha: float) -> np.ndarray:
    """Which of `m` tests Bonferroni rejects: a boolean array of shape `(m,)`, in the order of
    `pvals`, true where `p <= alpha / m`.

    Example:
        >>> bonferroni_reject([0.001, 0.2, 0.012], 0.05).tolist()
        [True, False, True]
    """
    # YOUR CODE HERE
    raise NotImplementedError


def holm_reject(pvals: Sequence[float], alpha: float) -> np.ndarray:
    """Which of `m` tests Holm's step-down procedure rejects: a boolean array of shape `(m,)`,
    in the order of `pvals`.

    Requirements, each of which is graded: with the p-values sorted ascending, the `i`-th
    smallest (counting from 0) is compared to `alpha / (m - i)` and rejected when it is at or
    below that; the procedure STOPS at the first comparison that fails, and nothing after it is
    rejected.

    Example:
        >>> holm_reject([0.04, 0.001, 0.03], 0.05).tolist()   # 0.001 passes, 0.03 fails: stop
        [False, True, False]
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_corrections() -> None:
    got = bonferroni_reject([0.001, 0.2, 0.012], 0.05)
    assert got.tolist() == [True, False, True], (
        f"with 3 tests the bar is 0.05 / 3 = 0.0167: 0.001 and 0.012 pass, 0.2 does not; got "
        f"{got.tolist()} — and the mask must follow the ORIGINAL order of the p-values"
    )
    held = holm_reject([0.04, 0.001, 0.03], 0.05)
    assert held.tolist() == [False, True, False], (
        f"sorted: 0.001 passes against 0.05/3, then 0.03 fails against 0.05/2 — Holm STOPS "
        f"there, so 0.04 is not rejected although it is below 0.05; got {held.tolist()}"
    )
    both = holm_reject([0.01, 0.2, 0.012, 0.9], 0.05)
    assert both.tolist() == [True, False, True, False], (
        f"0.01 passes 0.05/4, 0.012 passes 0.05/3, then 0.2 fails 0.05/2; got {both.tolist()}"
    )
    print("exercise 10 looks right — Bonferroni in one comparison, Holm with its stopping rule, "
          "both returning the mask in the original order")

In [ ]:
_try("exercise 10", _check_corrections)

## 17. Twenty metrics: false alarms and what the corrections cost

`FAMILY_SIMS` experiments, each reporting `FAMILY_M` metrics for `FAMILY_N` users per group.
Two scenarios on the same raw data. In the first, **no metric has a real effect**: every
rejection is a false alarm, and we measure how often at least one occurs (the family-wise
error rate) with no correction, with Bonferroni and with Holm. In the second, **two metrics
have a real relative lift** of `FAMILY_LIFT` and the rest do not: we measure how
often each procedure finds the real ones (power) and how often it still raises a false alarm.
In a third scenario `FAMILY_REAL_MANY` of them are real. The metrics are simulated independently
of one another, as the formula below assumes, and their p-values come from the normal approximation. The first scenario compares the measured rate
with the formula `1 - (1 - alpha) ** m` that assumes independent, perfectly calibrated tests.
This cell uses exercise 10.

In [ ]:
FAMILY_SIMS = 2000
FAMILY_M = 20
FAMILY_N = 200
FAMILY_REAL = 2
FAMILY_REAL_MANY = 10
FAMILY_LIFT = 0.30


def show_many_metrics() -> None:
    rng = make_rng("family")
    x = draw_population(rng, (FAMILY_SIMS, FAMILY_M, FAMILY_N))
    y = draw_population(rng, (FAMILY_SIMS, FAMILY_M, FAMILY_N))
    p_null = two_sided_p(welch_z(x, y))
    # Metrics are independent of each other, so one lifted pass serves both scenarios: the real
    # metrics' p-values are the first FAMILY_REAL (scenario 2) or FAMILY_REAL_MANY (scenario 3)
    # columns of it, and every other metric keeps its null p-value.
    p_lift = two_sided_p(welch_z(x[:, :FAMILY_REAL_MANY, :],
                                 (1.0 + FAMILY_LIFT) * y[:, :FAMILY_REAL_MANY, :]))
    p_real = np.concatenate([p_lift[:, :FAMILY_REAL], p_null[:, FAMILY_REAL:]], axis=1)
    p_many = np.concatenate([p_lift, p_null[:, FAMILY_REAL_MANY:]], axis=1)
    del x, y
    procedures = (("no correction", lambda p: p <= ALPHA),
                  ("Bonferroni", lambda p: bonferroni_reject(p, ALPHA)),
                  ("Holm", lambda p: holm_reject(p, ALPHA)))

    print(f"{FAMILY_SIMS} experiments, {FAMILY_M} metrics, {FAMILY_N} users per group, "
          f"nominal level {100 * ALPHA:.0f}%\n")
    print(f"scenario 1: none of the {FAMILY_M} metrics has a real effect")
    per_metric = float(np.mean(p_null <= ALPHA))
    formula = 1 - (1 - ALPHA) ** FAMILY_M
    per_se = math.sqrt(ALPHA * (1 - ALPHA) / p_null.size)
    print(f"  one metric alone is significant in {100 * per_metric:.1f}% of experiments "
          f"(MC SE {100 * per_se:.1f}% over all {p_null.size} null tests), "
          + ("below the nominal 5%: the normal formula is slightly conservative per metric here"
             if per_metric < ALPHA - 2 * per_se else
             "above the nominal 5%: the normal formula rejects too often per metric here"
             if per_metric > ALPHA + 2 * per_se else "consistent with the nominal 5%"))
    print(f"  the formula 1 - (1 - alpha)^m predicts {100 * formula:.1f}% for at least one of "
          f"{FAMILY_M}; with the measured single-metric rate in place of alpha it predicts "
          f"{100 * (1 - (1 - per_metric) ** FAMILY_M):.1f}%")
    fwer = {}
    for name, rule in procedures:
        hits = np.array([rule(row).any() for row in p_null])
        fwer[name] = float(hits.mean())
        se = math.sqrt(fwer[name] * (1 - fwer[name]) / FAMILY_SIMS)
        print(f"  {name:<14} at least one false alarm in {100 * fwer[name]:>5.1f}% of "
              f"experiments (MC SE {100 * se:.1f}%)")

    print(f"\nscenario 2: the first {FAMILY_REAL} metrics have a {100 * FAMILY_LIFT:.0f}% lift")
    print(f"  {'procedure':<14} {'power on the real ones':>23} {'any false alarm among the rest':>32}")
    power = {}
    for name, rule in procedures:
        masks = np.array([rule(row) for row in p_real])
        power[name] = float(masks[:, :FAMILY_REAL].mean())
        false_alarm = float(masks[:, FAMILY_REAL:].any(axis=1).mean())
        print(f"  {name:<14} {100 * power[name]:>22.1f}% {100 * false_alarm:>31.1f}%")
    print(f"\nscenario 3: the first {FAMILY_REAL_MANY} metrics have a {100 * FAMILY_LIFT:.0f}% lift")
    many = {}
    for name, rule in procedures:
        masks = np.array([rule(row) for row in p_many])
        many[name] = float(masks[:, :FAMILY_REAL_MANY].mean())
        print(f"  {name:<14} power on the real ones {100 * many[name]:>5.1f}%")
    print(f"\nscenario 2: Bonferroni costs {100 * (power['no correction'] - power['Bonferroni']):.1f} "
          f"points of power against doing nothing, and Holm recovers "
          f"{100 * (power['Holm'] - power['Bonferroni']):.1f} of them.")
    gain2 = power["Holm"] - power["Bonferroni"]
    gain3 = many["Holm"] - many["Bonferroni"]
    print(f"scenario 3: Bonferroni costs {100 * (many['no correction'] - many['Bonferroni']):.1f} "
          f"points and Holm recovers {100 * gain3:.1f}.")
    if gain3 > gain2:
        print("Holm's advantage is larger when more of the metrics are real: every rejection it "
              "makes lowers the bar for the next one. Its family-wise guarantee is the same as "
              "Bonferroni's.")
    else:
        print("Holm's advantage is not larger here when more of the metrics are real; with these "
              "settings the two procedures stay close.")
    _RESULTS["family_formula"] = formula
    _RESULTS["family_none"] = fwer["no correction"]
    _RESULTS["family_bonferroni"] = fwer["Bonferroni"]
    _RESULTS["family_holm"] = fwer["Holm"]


_demo("many metrics demo", show_many_metrics, needs=("exercise 10",))

## 18. Common mistakes

- **Treating "95%" as a property of one interval.** It describes the procedure. Section 8
  measured how often that procedure was right; the answer depends on the statistic and on how
  much data there is.
- **`std` without `ddof=1`.** A standard error estimated from a simulation divides by
  `reps - 1`, and every later number inherits the choice.
- **Resampling without replacement.** A resample that never repeats an observation is just the
  sample shuffled, so every statistic that ignores order comes out identical, and the
  "interval" has zero width.
- **Fitting the log-log slope through the origin.** The constant in `c * n**k` is not 1.
- **A p-value of exactly 0.** A permutation test with `n_perm` shuffles cannot resolve a p-value
  below `1 / (n_perm + 1)`; report that floor, never zero.
- **Counting ties as not extreme.** `>` where `>=` was meant undercounts every tie, and on small
  or discrete data that is a visibly different p-value.
- **Comparing floats with `==`.** Two shuffles equal in arithmetic can differ in the last binary
  digit; compare with a tolerance.
- **Adding a lift instead of multiplying.** A relative lift scales the whole distribution,
  its tail included, which is what makes a heavy-tailed metric hard to move.
- **Peeking "just to check".** Every look at a running test is an extra test. If the plan is
  to stop early, the test has to be designed for it; the plain p-value is not.
- **Correcting in the wrong order.** Holm stops at its first failure; testing every sorted
  p-value independently against its own threshold rejects things Holm must not.

## 19. What the lesson measured

Every figure below was computed by the demos you ran above; any a demo did not reach is
named rather than guessed.

In [ ]:
def _scorecard() -> None:
    r = _RESULTS
    lines = (
        ("lowest bootstrap coverage, mean", "cover_mean_lowest", f"nominal {100 * LEVEL:.0f}%"),
        ("lowest bootstrap coverage, median", "cover_median_lowest", f"nominal {100 * LEVEL:.0f}%"),
        ("permutation test, no real effect", "null_perm_rate", f"nominal {100 * ALPHA:.0f}%"),
        ("Welch z, no real effect", "null_z_rate", f"nominal {100 * ALPHA:.0f}%"),
        ("one look at the end", "peek_final", f"nominal {100 * ALPHA:.0f}%"),
        ("stop at the first significant look", "peek_any", f"nominal {100 * ALPHA:.0f}%"),
        (f"{FAMILY_M} metrics, no correction", "family_none", f"formula {100 * r.get('family_formula', float('nan')):.1f}%"),
        (f"{FAMILY_M} metrics, Bonferroni", "family_bonferroni", f"nominal {100 * ALPHA:.0f}%"),
        (f"{FAMILY_M} metrics, Holm", "family_holm", f"nominal {100 * ALPHA:.0f}%"),
    )
    missing = []
    print(f"{'what was measured':<38} {'measured':>9}   against")
    for label, key, against in lines:
        if key in r:
            print(f"{label:<38} {100 * r[key]:>8.1f}%   {against}")
        else:
            missing.append(label)
    if missing:
        print("\nnot measured yet (finish the exercises above and re-run the cells): "
              + "; ".join(missing))


_demo("scorecard", _scorecard)

## 20. Self-check

1. A teammate checks a running A/B test every day and ships the first time p < 0.05. Compared
   with one look at a pre-planned sample size, what happens to the false-positive rate when
   the treatment does nothing?
   - (a) it stays at 5%, because each individual look is still a 5% test
   - (b) it rises above 5%, because every extra look is another chance for noise to cross the
     line
   - (c) it falls below 5%, because stopping early means less data to be misled by

2. A permutation test with 999 shuffles finds none as extreme as the observed difference.
   The p-value to report is:
   - (a) exactly 0, since nothing was as extreme
   - (b) 0.05, the usual threshold
   - (c) 1/1000, because the observed arrangement counts as one of the possible ones

3. You simulate a 95% bootstrap interval for a mean on a heavy-tailed population and it
   covers the truth in 91% of 2000 trials, with a Monte-Carlo standard error under one point.
   What does that tell you?
   - (a) the procedure under-covers at that sample size: its "95%" overstates the confidence
   - (b) nothing, because 91% is within noise of 95%
   - (c) the interval is correct, but each individual interval is too wide

4. Holm and Bonferroni both control the chance of any false alarm across m tests at alpha.
   Which statement is true?
   - (a) Bonferroni rejects everything Holm rejects, and sometimes more
   - (b) they always reject exactly the same tests
   - (c) Holm rejects everything Bonferroni rejects, and sometimes more

5. A power simulation says 78% power at your planned sample size, with a Monte-Carlo standard
   error of about 3 points, and your target is 80%. The honest reading is:
   - (a) the plan is safe, since 78% rounds to 80%
   - (b) the difference from 80% is within simulation noise, so the plan is borderline and
     deserves more simulations or a larger sample
   - (c) the experiment is underpowered and must be cancelled

Answers come with this lesson's worked solution when you enrol on Synapsa.

## What you built

A sampling-distribution simulator and the log-log fit that reads a rate off it; a bootstrap
interval and the coverage audit that tells you how far to trust it; a permutation test with the
tie and add-one details that keep its p-value honest; a power estimator and the sample-size
rule that uses it; and two audits, peeking and many metrics, with the corrections that repair
them. Every claim a result makes about a "5%" or a "95%" is checkable by this kind of
simulation, and now you can check it.

In [ ]:
# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    _ALL_CHECKS = (  # (exercise, its check, the exercises that check relies on)
        ("exercise 1", _check_simulate_sampling_distribution, ()),
        ("exercise 2", _check_se_and_slope, ()),
        ("exercise 3", _check_bootstrap, ()),
        ("exercise 4", _check_coverage_rate, ()),
        ("exercise 5", _check_permutation_test, ()),
        ("exercise 6", _check_estimate_power, ()),
        ("exercise 7", _check_smallest_n, ()),
        ("exercise 8", _check_z_at_looks, ()),
        ("exercise 9", _check_rejection_rates, ()),
        ("exercise 10", _check_corrections, ()),
    )
    import contextlib
    import io
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check, _needs in _ALL_CHECKS:
            _try(_name, _check, needs=_needs)
    _progress_board()
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends the run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.